# Разбор A. Структурное сходство

Ксантины имеют общий мотив, но список «похожих» зависит от определения сходства. Проверим, насколько рейтинг к кофеину устойчив к радиусу Morgan. Гипотеза: локальная похожесть не совпадает с равенством молекул. Для B кислота и лактон могут иметь высокий коэффициент, оставаясь разными аналитическими объектами.

Получите fingerprints для четырёх ксантинов при radius=2, chirality=True. Вычислите полную матрицу, проверьте симметрию и единицы на диагонали. Измените радиус на 3 и сравните рейтинг. Отдельный ручной пример двух множеств проверяет формулу вне RDKit.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M05-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from rdkit import Chem,DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
names=['caffeine','theophylline','theobromine','istradefylline']
def matrix(radius):
    g=rdFingerprintGenerator.GetMorganGenerator(radius=radius,fpSize=2048,includeChirality=True)
    fps=[g.GetFingerprint(Chem.MolFromSmiles(compound(n)['smiles'])) for n in names]
    return np.array([[DataStructs.TanimotoSimilarity(a,b) for b in fps] for a in fps])
values=matrix(2);assert np.allclose(values,values.T) and np.allclose(np.diag(values),1)
display(values)


array([[1.        , 0.45714286, 0.52941176, 0.22033898],
       [0.45714286, 1.        , 0.29268293, 0.19672131],
       [0.52941176, 0.29268293, 1.        , 0.17460317],
       [0.22033898, 0.19672131, 0.17460317, 1.        ]])

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,ax=plt.subplots();im=ax.imshow(values,vmin=0,vmax=1);fig.colorbar(im,ax=ax,label='Tanimoto')
ax.set_xticks(range(4),names,rotation=25);ax.set_yticks(range(4),names);publish_plot(fig,'similarity.png')
print('radius=3 caffeine similarities',matrix(3)[0].tolist())


<Figure size 640x480 with 2 Axes>

radius=3 caffeine similarities [1.0, 0.3333333333333333, 0.4, 0.15294117647058825]


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
a={1,2,3};b={2,3,4,5};assert len(a&b)/len(a|b)==.4
result={'names':names,'matrix':values.tolist(),'kind':'calculated','radius':2,'fpSize':2048,'includeChirality':True}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M05-L01','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\practicuum\evidence\20261119-main-unification\notebooks\workspace\my_research\evidence\M05-L01\20261119T123251258247Z-A-cad1e49efbe24efb8063382bb58e5e2a.json


Изменение рейтинга является свойством представления. Чтобы выбрать следующий опыт, нужно учитывать не только сходство, но и неизвестность. Следующий урок добавит несколько критериев отбора.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?